# Confidence Intervals for Accuracy

This tutorial uses synthetic **Project Frog** evaluation results to answer a practical question: when should an engineer trust an observed accuracy metric?

We focus on a fixed expected accuracy of **0.85** and vary sample size to show two ideas:

1. A smaller sample lowers confidence in the estimate.
2. A smaller sample does **not** automatically lower the true accuracy.

## Environment setup

Run this notebook from the repository root after `uv sync`. It expects the local `applied_stats_ai` package plus NumPy, Pandas, Matplotlib, and Seaborn from the project environment.

Typical workflow:
- `python -m uv sync`
- `python -m uv run jupyter lab tutorials/confidence_intervals/confidence_intervals_for_accuracy.ipynb`


## Why does an AI engineer need to understand this?

Release dashboards often report a single number such as accuracy = 0.84. Without uncertainty, that number can be misleading. In Project Frog, engineers need to know whether a metric moved because the system changed or because the sample is too small to pin performance down precisely.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from applied_stats_ai import bootstrap_metric, clopper_pearson_interval, wilson_interval

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(7)
accuracy = 0.85
sample_sizes = [20, 50, 100, 500, 1000]

## Accuracy as a binomial proportion

If each Project Frog evaluation outcome is marked correct or incorrect, then observed accuracy is the proportion of successes in a binomial-style sample. That gives us a simple way to reason about standard error and interval estimates.

In [ ]:
outcomes = {n: rng.binomial(1, accuracy, size=n) for n in sample_sizes}

summary_rows = []
for n, values in outcomes.items():
    successes = int(values.sum())
    observed_accuracy = successes / n
    standard_error = math.sqrt(observed_accuracy * (1 - observed_accuracy) / n)
    wilson = wilson_interval(successes, n)
    exact = clopper_pearson_interval(successes, n)
    summary_rows.append(
        {
            "sample_size": n,
            "successes": successes,
            "observed_accuracy": observed_accuracy,
            "standard_error": standard_error,
            "wilson_lower": wilson[0],
            "wilson_upper": wilson[1],
            "exact_lower": exact[0],
            "exact_upper": exact[1],
        }
    )

summary = pd.DataFrame(summary_rows)
summary.round(4)

## Wilson and Clopper-Pearson intervals

Wilson intervals usually behave better than the simple Wald interval, especially on smaller samples. Clopper-Pearson intervals are exact for the binomial model and often a bit more conservative.

In [ ]:
display_columns = [
    "sample_size",
    "observed_accuracy",
    "standard_error",
    "wilson_lower",
    "wilson_upper",
    "exact_lower",
    "exact_upper",
]
summary[display_columns].round(4)

## Bootstrap intervals

A bootstrap interval is useful when you want a computational estimate of uncertainty from the observed outcomes. It does not change the underlying truth; it summarizes how unstable the estimate could be under repeated resampling from the observed sample.

In [ ]:
bootstrap_rows = []
for n, values in outcomes.items():
    result = bootstrap_metric(values, np.mean, n_resamples=4000, random_state=n)
    bootstrap_rows.append(
        {
            "sample_size": n,
            "bootstrap_estimate": result["estimate"],
            "bootstrap_lower": result["lower"],
            "bootstrap_upper": result["upper"],
        }
    )

bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.round(4)

## Visualization of uncertainty

The next chart keeps the expected generating accuracy fixed at **0.85** and shows how interval width shrinks as sample size grows.

In [ ]:
plot_df = summary.merge(bootstrap_summary, on="sample_size")
plot_df["wilson_width"] = plot_df["wilson_upper"] - plot_df["wilson_lower"]
plot_df["exact_width"] = plot_df["exact_upper"] - plot_df["exact_lower"]
plot_df["bootstrap_width"] = plot_df["bootstrap_upper"] - plot_df["bootstrap_lower"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(plot_df["sample_size"], plot_df["wilson_width"], marker="o", label="Wilson width")
ax.plot(plot_df["sample_size"], plot_df["exact_width"], marker="o", label="Clopper-Pearson width")
ax.plot(plot_df["sample_size"], plot_df["bootstrap_width"], marker="o", label="Bootstrap width")
ax.set_xscale("log")
ax.set_xlabel("Sample size (log scale)")
ax.set_ylabel("Interval width")
ax.set_title("Uncertainty shrinks as Project Frog evaluation size increases")
ax.legend()
plt.show()

## Sample-size effects and the common misconception

The true generating accuracy remains fixed at 0.85 in every simulation below. Only the **observed variability** changes.

In [ ]:
simulation_rows = []
for n in sample_sizes:
    repeated = rng.binomial(n, accuracy, size=2500) / n
    simulation_rows.extend({"sample_size": n, "observed_accuracy": value} for value in repeated)

sim_df = pd.DataFrame(simulation_rows)

fig, axes = plt.subplots(len(sample_sizes), 1, figsize=(8, 12), sharex=True)
for index, n in enumerate(sample_sizes):
    ax = axes[index]
    subset = sim_df.loc[sim_df["sample_size"] == n, "observed_accuracy"]
    sns.histplot(subset, bins=20, ax=ax, color="#2a9d8f")
    ax.axvline(accuracy, color="#e76f51", linestyle="--", label="Expected accuracy = 0.85")
    ax.set_title(f"Observed accuracy over repeated samples (n={n})")
    ax.legend()
plt.tight_layout()
plt.show()

## Interpretation

- Smaller samples make the estimate noisier and the intervals wider.
- The center of the repeated sampling distribution still stays near 0.85.
- Small samples reduce confidence in the estimate, but they do **not** automatically reduce the true underlying accuracy.
- If Project Frog reports 84% on 75 examples, the next question is not only *"is 84 lower?"* but also *"how uncertain is 84?"*

In [ ]:
interval_comparison = pd.DataFrame(
    {
        "sample_size": plot_df["sample_size"],
        "observed_accuracy": plot_df["observed_accuracy"],
        "wilson_interval": [
            (plot_df["wilson_lower"].round(4).iloc[i], plot_df["wilson_upper"].round(4).iloc[i])
            for i in range(len(plot_df))
        ],
        "exact_interval": [
            (plot_df["exact_lower"].round(4).iloc[i], plot_df["exact_upper"].round(4).iloc[i])
            for i in range(len(plot_df))
        ],
        "bootstrap_interval": [
            (
                plot_df["bootstrap_lower"].round(4).iloc[i],
                plot_df["bootstrap_upper"].round(4).iloc[i],
            )
            for i in range(len(plot_df))
        ],
    }
)
interval_comparison


## Next questions for an engineer

Once interval estimation is clear, the next engineering questions are:

- Are two releases being compared on the same examples?
- Is the case mix comparable?
- Are observations independent, or clustered by project?
- What additional data would reduce uncertainty enough to change the release decision?